# Cycle B Analysis — Questerix Fractions Playtest

**Purpose:** Analyze pre-test, post-test, and in-app session data to validate whether Questerix Fractions teaches fraction concepts to K–2 students.

**Data source:** `validation-data/cycle-b/`

**Expected outputs:**
- Mean pre/post-test scores and gains
- Statistical comparison (t-test, effect size)
- Hypothesis result: learning occurred (SUPPORTED) or no learning (UNSUPPORTED)
- Visualizations (pre vs post, gain distribution)
- Breakdown by grade, device, and session progression
- Misconception rate analysis (optional)

---

## Setup & Data Import

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

# Set up paths
data_dir = Path('./validation-data/cycle-b')
json_dir = data_dir / 'json-exports'

# Load results (pre/post scores)
results = pd.read_csv(data_dir / 'results.csv')
print(f'Loaded {len(results)} participants')
print(results.head())

In [ ]:
# Load session metadata
sessions = pd.read_csv(data_dir / 'sessions.csv')
print(f'Loaded {len(sessions)} sessions')
print(sessions.head())

## Part 1: Descriptive Statistics — Pre & Post Test

In [ ]:
# Cohort summary
print('=== COHORT SUMMARY ===')
print(f'N = {len(results)}')
print(f'Grade distribution:')
print(results['grade'].value_counts().sort_index())
print(f'\nDevice distribution:')
print(results['device'].value_counts())

In [ ]:
# Pre-test statistics
print('=== PRE-TEST SCORES ===')
print(f'Mean: {results["pre_test_score"].mean():.2f} / 8')
print(f'SD: {results["pre_test_score"].std():.2f}')
print(f'Min: {results["pre_test_score"].min()}')
print(f'Max: {results["pre_test_score"].max()}')
print(f'Median: {results["pre_test_score"].median():.1f}')

In [ ]:
# Post-test statistics
print('=== POST-TEST SCORES ===')
print(f'Mean: {results["post_test_score"].mean():.2f} / 8')
print(f'SD: {results["post_test_score"].std():.2f}')
print(f'Min: {results["post_test_score"].min()}')
print(f'Max: {results["post_test_score"].max()}')
print(f'Median: {results["post_test_score"].median():.1f}')

In [ ]:
# Gain statistics
print('=== LEARNING GAIN (Post − Pre) ===')
print(f'Mean gain: {results["gain"].mean():.2f} points')
print(f'SD: {results["gain"].std():.2f}')
print(f'Min gain: {results["gain"].min()}')
print(f'Max gain: {results["gain"].max()}')
print(f'Median gain: {results["gain"].median():.1f}')
print(f'\nParticipants with gain ≥ 3 points: {(results["gain"] >= 3).sum()} / {len(results)}')
print(f'Participants with gain ≥ 0 (no decline): {(results["gain"] >= 0).sum()} / {len(results)}')

## Part 2: Statistical Test — Pre vs Post

In [ ]:
# Paired t-test: pre vs post (assumes normal distribution)
t_stat, p_value = stats.ttest_rel(results['post_test_score'], results['pre_test_score'])

print('=== PAIRED T-TEST (Pre vs Post) ===')
print(f't-statistic: {t_stat:.3f}')
print(f'p-value: {p_value:.4f}')
print(f'Significance: ', end='')
if p_value < 0.05:
    print(f'YES (p < 0.05) — Learning is statistically significant')
elif p_value < 0.10:
    print(f'MARGINAL (p < 0.10) — Suggestive but not definitive')
else:
    print(f'NO (p ≥ 0.05) — No significant difference detected')

In [ ]:
# Wilcoxon signed-rank test (non-parametric alternative, robust to outliers)
w_stat, p_value_wilcox = stats.wilcoxon(results['post_test_score'], results['pre_test_score'])

print('=== WILCOXON SIGNED-RANK TEST (Pre vs Post) ===')
print(f'W-statistic: {w_stat:.3f}')
print(f'p-value: {p_value_wilcox:.4f}')
print(f'Significance: ', end='')
if p_value_wilcox < 0.05:
    print(f'YES (p < 0.05) — Learning is statistically significant (non-parametric test)')
else:
    print(f'NO (p ≥ 0.05) — No significant difference detected (non-parametric test)')

In [ ]:
# Effect size (Cohen's d for paired data)
def cohens_d_paired(pre, post):
    diff = post - pre
    return diff.mean() / diff.std()

d = cohens_d_paired(results['pre_test_score'], results['post_test_score'])

print('=== EFFECT SIZE (Cohen\'s d) ===')
print(f"Cohen's d: {d:.3f}")
print(f'Interpretation: ', end='')
if abs(d) < 0.2:
    print('negligible')
elif abs(d) < 0.5:
    print('small')
elif abs(d) < 0.8:
    print('medium')
else:
    print('large')

## Part 3: Hypothesis Test

In [ ]:
# Success criterion: mean gain ≥ 3 points
success_threshold = 3
mean_gain = results['gain'].mean()
se_gain = results['gain'].std() / np.sqrt(len(results))
ci_lower = mean_gain - 1.96 * se_gain
ci_upper = mean_gain + 1.96 * se_gain

print('=== PRIMARY HYPOTHESIS ===')
print(f'Hypothesis: Post-test gain ≥ {success_threshold} points indicates learning')
print(f'\nObserved mean gain: {mean_gain:.2f} points')
print(f'95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]')
print(f'\nResult: ', end='')
if mean_gain >= success_threshold:
    print(f'SUPPORTED — Mean gain {mean_gain:.2f} ≥ {success_threshold} points')
elif ci_lower >= success_threshold:
    print(f'SUPPORTED — Even lower bound of CI [{ci_lower:.2f}] exceeds {success_threshold} points')
else:
    print(f'UNSUPPORTED — Mean gain {mean_gain:.2f} < {success_threshold} points')

## Part 4: Visualizations

In [ ]:
# Pre vs Post distribution (box plot)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Box plot
data_to_plot = [results['pre_test_score'], results['post_test_score']]
ax1.boxplot(data_to_plot, labels=['Pre-Test', 'Post-Test'])
ax1.set_ylabel('Score (0–8 points)')
ax1.set_title('Pre-Test vs Post-Test Distribution')
ax1.grid(axis='y', alpha=0.3)

# Scatter plot with identity line
ax2.scatter(results['pre_test_score'], results['post_test_score'], s=80, alpha=0.6)
min_val = 0
max_val = 8
ax2.plot([min_val, max_val], [min_val, max_val], 'k--', label='No change', alpha=0.5)
ax2.set_xlabel('Pre-Test Score')
ax2.set_ylabel('Post-Test Score')
ax2.set_title('Individual Trajectories')
ax2.set_xlim(-0.5, 8.5)
ax2.set_ylim(-0.5, 8.5)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(data_dir / 'pre_vs_post.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Gain distribution
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(results['gain'], bins=range(0, int(results['gain'].max()) + 2), 
        color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(results['gain'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean = {results["gain"].mean():.2f}')
ax.axvline(3, color='green', linestyle='--', linewidth=2, label='Success threshold = 3')
ax.set_xlabel('Learning Gain (Post − Pre, points)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Learning Gains')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(data_dir / 'gain_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 5: Breakdown by Grade

In [ ]:
# Pre/post by grade
by_grade = results.groupby('grade')[['pre_test_score', 'post_test_score', 'gain']].agg(['mean', 'std', 'count'])

print('=== PERFORMANCE BY GRADE ===')
for grade in sorted(results['grade'].unique()):
    grade_data = results[results['grade'] == grade]
    print(f'\nGrade {grade} (N={len(grade_data)})')
    print(f'  Pre-test:  {grade_data["pre_test_score"].mean():.2f} ± {grade_data["pre_test_score"].std():.2f}')
    print(f'  Post-test: {grade_data["post_test_score"].mean():.2f} ± {grade_data["post_test_score"].std():.2f}')
    print(f'  Gain:      {grade_data["gain"].mean():.2f} ± {grade_data["gain"].std():.2f}')

## Part 6: Breakdown by Device

In [ ]:
# Pre/post by device
print('=== PERFORMANCE BY DEVICE ===')
for device in sorted(results['device'].unique()):
    device_data = results[results['device'] == device]
    print(f'\n{device} (N={len(device_data)})')
    print(f'  Pre-test:  {device_data["pre_test_score"].mean():.2f} ± {device_data["pre_test_score"].std():.2f}')
    print(f'  Post-test: {device_data["post_test_score"].mean():.2f} ± {device_data["post_test_score"].std():.2f}')
    print(f'  Gain:      {device_data["gain"].mean():.2f} ± {device_data["gain"].std():.2f}')

## Part 7: Session-to-Session Progression (Optional)

In [ ]:
# If session JSONs are available, analyze progression across 3 sessions
# This is a placeholder for extracting session-level metrics

print('=== SESSION PROGRESSION ===')
print('(To be filled in when session JSON analysis is complete)')
print('\nPossible metrics to extract:')
print('- Correct answer rate per session (Session 1 vs Session 3)')
print('- Session duration and completion rate')
print('- Time-to-correct per question (learning efficiency)')
print('- Misconception recovery rate (attempts before correct answer)')

## Part 8: Misconception Analysis (Optional)

In [ ]:
# Check specific misconception rates (from learning hypotheses)
print('=== MISCONCEPTION ANALYSIS ===')
print('(To be filled when pre/post-test item-level data is available)')
print('\nPossible misconceptions to track:')
print('- MC-WHB-01 (whole-number bias): larger numerator = larger fraction')
print('- MC-WHB-02 (denominator bias): confusing denominator with fraction size')
print('- MC-EOL-01 (equal-parts logic): assumption that divided pieces are automatically equal')
print('\nIf item-level data available, compute:')
print('- Misconception rate at pre-test (% students with misconception)')
print('- Misconception rate at post-test')
print('- Recovery rate (% who corrected the misconception)')

## Part 9: Summary Report

In [ ]:
# Generate summary statistics
summary = f"""
        
        =================================================
        CYCLE B ANALYSIS SUMMARY
        =================================================
        
        COHORT
        ------
        Participants: {len(results)}
        Grade distribution: {dict(results['grade'].value_counts().sort_index())}
        Device distribution: {dict(results['device'].value_counts())}
        
        PRE-TEST
        --------
        Mean: {results['pre_test_score'].mean():.2f} / 8.0
        SD: {results['pre_test_score'].std():.2f}
        Range: [{results['pre_test_score'].min()}, {results['pre_test_score'].max()}]
        
        POST-TEST
        ---------
        Mean: {results['post_test_score'].mean():.2f} / 8.0
        SD: {results['post_test_score'].std():.2f}
        Range: [{results['post_test_score'].min()}, {results['post_test_score'].max()}]
        
        LEARNING GAIN (Post − Pre)
        --------------------------
        Mean gain: {results['gain'].mean():.2f} points
        SD: {results['gain'].std():.2f}
        Range: [{results['gain'].min()}, {results['gain'].max()}]
        Participants with gain ≥ 3: {(results['gain'] >= 3).sum()} / {len(results)}
        
        STATISTICAL TEST
        -----------------
        Paired t-test: t({len(results)-1}) = {t_stat:.3f}, p = {p_value:.4f}
        Wilcoxon test: W = {w_stat:.3f}, p = {p_value_wilcox:.4f}
        Cohen's d: {d:.3f}
        
        PRIMARY HYPOTHESIS
        ------------------
        H0: Post-test gain < 3 points (no learning)
        H1: Post-test gain ≥ 3 points (learning occurred)
        
        Result: Mean observed gain = {results['gain'].mean():.2f} points
        Conclusion: {'SUPPORTED' if results['gain'].mean() >= 3 else 'UNSUPPORTED'}
        
        =================================================
        """

        print(summary)
        
        # Save to file
        with open(data_dir / 'summary.txt', 'w') as f:
            f.write(summary)

## Part 10: Export for Report

In [ ]:
# Create a clean CSV for the final report
report_data = results[['pseudonym', 'pre_test_score', 'post_test_score', 'gain', 'grade', 'device']].copy()
report_data.to_csv(data_dir / 'report_data.csv', index=False)

print('Report data exported to: report_data.csv')
print(report_data)

---

## Next Steps (Phase 4)

1. **Item-level analysis:** If pre/post-test responses are available by item, analyze which items showed the most improvement
2. **Session JSON analysis:** Extract interaction patterns from session JSONs to understand learning trajectories
3. **Misconception tracking:** Check if specific misconceptions (WHB, EOL) were corrected
4. **Observer notes synthesis:** Qualitatively analyze observer notes for themes (engagement, confusion, breakthroughs)
5. **Report writing:** Summarize findings in `report.md` for stakeholders
6. **Recommendations:** Propose changes for Phase 3 based on findings

---

**Analysis completed:** [TODAY's DATE]  
**Analyst:** [YOUR NAME]  
**Data retention:** All results archived; individual data deleted per schedule (see `cycle-b/README.md`)
